# Librerias

In [ ]:
import os
import sqlite3
import time
import pandas as pd
import requests
from dotenv import load_dotenv



# Configuraciones

In [2]:
load_dotenv()

TWITCH_CLIENT_ID = os.getenv('TWITCH_CLIENT_ID')
TWITCH_CLIENT_SECRET = os.getenv('TWITCH_CLIENT_SECRET')
DB_PATH = 'gaming_warehouse.db'
_token_cache = {"token": None, "Expira": 0}


# Definir base de datos

In [ ]:
def inicializar_base_datos():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("PRAGMA foreign_keys = ON;")
    cursor.execute("PRAGMA journal_mode = WAL;")

    esquema_sql = """
    --Nuestra tabla principal de juegos
    CREATE TABLE IF NOT EXISTS CAT_Juego (
        juego_id INTEGER PRIMARY KEY AUTOINCREMENT,
        id_igdb INTEGER UNIQUE,
        id_steam INTEGER UNIQUE,
        titulo TEXT NOT NULL,
        categoria INTEGER,
        fecha_lanzamiento DATE,
        resumen TEXT,
        historia TEXT,
        url_portada TEXT,
        puntuacion_igdb REAL,
        conteo_votos_igdb INTEGER,
        conteo_dlc INTEGER,
        conteo_videos INTEGER,
        hltb_historia_principal REAL,
        hltb_historia_extra REAL,
        hltb_completacionista REAL
    );

    -- Relacion del juego con sus DLCs
    CREATE TABLE IF NOT EXISTS REL_Juego_DLC (
        juego_id_principal INTEGER,
        juego_id_dlc INTEGER,
        PRIMARY KEY (juego_id_principal, juego_id_dlc),
        FOREIGN KEY (juego_id_principal) REFERENCES CAT_Juego(juego_id) ON DELETE CASCADE,
        FOREIGN KEY (juego_id_dlc) REFERENCES CAT_Juego(juego_id) ON DELETE CASCADE
    );

    -- Aqui eetsna nuestros catalogos que podemos obtener de los juegos y que conviene tenerlos separados
    CREATE TABLE IF NOT EXISTS CAT_Genero (
        genero_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE
    );

    CREATE TABLE IF NOT EXISTS CAT_Tematica (
        tematica_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE
    );

    CREATE TABLE IF NOT EXISTS CAT_Modo_Juego (
        modo_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE
    );

    CREATE TABLE IF NOT EXISTS CAT_Perspectiva (
        perspectiva_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE
    );

    CREATE TABLE IF NOT EXISTS CAT_Empresa (
        empresa_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE
    );

    CREATE TABLE IF NOT EXISTS CAT_Franquicia (
        franquicia_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE
    );

    CREATE TABLE IF NOT EXISTS CAT_Etiqueta (
        etiqueta_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE
    );

    CREATE TABLE IF NOT EXISTS CAT_Plataforma (
        plataforma_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE
    );

    -- Tablas de relacion ya que mi base tiene una relacion de muchos amuchos
    CREATE TABLE IF NOT EXISTS REL_Juego_Genero (
        juego_id INTEGER,
        genero_id INTEGER,
        PRIMARY KEY (juego_id, genero_id),
        FOREIGN KEY (juego_id) REFERENCES CAT_Juego(juego_id) ON DELETE CASCADE,
        FOREIGN KEY (genero_id) REFERENCES CAT_Genero(genero_id) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS REL_Juego_Tematica (
        juego_id INTEGER,
        tematica_id INTEGER,
        PRIMARY KEY (juego_id, tematica_id),
        FOREIGN KEY (juego_id) REFERENCES CAT_Juego(juego_id) ON DELETE CASCADE,
        FOREIGN KEY (tematica_id) REFERENCES CAT_Tematica(tematica_id) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS REL_Juego_Modo (
        juego_id INTEGER,
        modo_id INTEGER,
        PRIMARY KEY (juego_id, modo_id),
        FOREIGN KEY (juego_id) REFERENCES CAT_Juego(juego_id) ON DELETE CASCADE,
        FOREIGN KEY (modo_id) REFERENCES CAT_Modo_Juego(modo_id) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS REL_Juego_Perspectiva (
        juego_id INTEGER,
        perspectiva_id INTEGER,
        PRIMARY KEY (juego_id, perspectiva_id),
        FOREIGN KEY (juego_id) REFERENCES CAT_Juego(juego_id) ON DELETE CASCADE,
        FOREIGN KEY (perspectiva_id) REFERENCES CAT_Perspectiva(perspectiva_id) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS REL_Juego_Desarrollador (
        juego_id INTEGER,
        empresa_id INTEGER,
        PRIMARY KEY (juego_id, empresa_id),
        FOREIGN KEY (juego_id) REFERENCES CAT_Juego(juego_id) ON DELETE CASCADE,
        FOREIGN KEY (empresa_id) REFERENCES CAT_Empresa(empresa_id) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS REL_Juego_Editor (
        juego_id INTEGER,
        empresa_id INTEGER,
        PRIMARY KEY (juego_id, empresa_id),
        FOREIGN KEY (juego_id) REFERENCES CAT_Juego(juego_id) ON DELETE CASCADE,
        FOREIGN KEY (empresa_id) REFERENCES CAT_Empresa(empresa_id) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS REL_Juego_Franquicia (
        juego_id INTEGER,
        franquicia_id INTEGER,
        PRIMARY KEY (juego_id, franquicia_id),
        FOREIGN KEY (juego_id) REFERENCES CAT_Juego(juego_id) ON DELETE CASCADE,
        FOREIGN KEY (franquicia_id) REFERENCES CAT_Franquicia(franquicia_id) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS REL_Juego_Etiqueta (
        juego_id INTEGER,
        etiqueta_id INTEGER,
        PRIMARY KEY (juego_id, etiqueta_id),
        FOREIGN KEY (juego_id) REFERENCES CAT_Juego(juego_id) ON DELETE CASCADE,
        FOREIGN KEY (etiqueta_id) REFERENCES CAT_Etiqueta(etiqueta_id) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS REL_Juego_Plataforma (
        juego_id INTEGER,
        plataforma_id INTEGER,
        PRIMARY KEY (juego_id, plataforma_id),
        FOREIGN KEY (juego_id) REFERENCES CAT_Juego(juego_id) ON DELETE CASCADE,
        FOREIGN KEY (plataforma_id) REFERENCES CAT_Plataforma(plataforma_id) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS REL_Juegos_Similares (
        juego_id INTEGER,
        id_igdb_similar INTEGER,
        PRIMARY KEY (juego_id, id_igdb_similar),
        FOREIGN KEY (juego_id) REFERENCES CAT_Juego(juego_id) ON DELETE CASCADE
    );
    """

    cursor.executescript(esquema_sql)
    conn.commit()
    conn.close()


In [4]:
inicializar_base_datos()

# Extraccion desde IGDB

In [ ]:
#mi ffuncion apra obtener el token de igdb, con cache incluido para no pedirlo cada vez que se necesite.
def obtener_token_igdb():
    ahora = time.time()

    if _token_cache["token"] and ahora < _token_cache["Expira"]:
        return _token_cache["token"]

    respuesta = requests.post(
        "https://id.twitch.tv/oauth2/token",
        params={
            "client_id": TWITCH_CLIENT_ID,
            "client_secret": TWITCH_CLIENT_SECRET,
            "grant_type": "client_credentials"
        }
    )
    respuesta.raise_for_status()

    datos = respuesta.json()
    _token_cache["token"] = datos["access_token"]
    _token_cache["Expira"] = ahora + datos.get("expires_in", 3600) - 300

    print("Token de IGDB actualizado exitosamente")
    return _token_cache["token"]



In [ ]:
# la duncion para descargar los juegos desde IGDB
def descargar_y_limpiar_juegos(paginas=1, juegos_por_pagina=500):
    #funcion para extraer los nombres de los catalogos
    def extraer_nombres(lista):
        if not isinstance(lista, list): return []
        return [item['name'] for item in lista if 'name' in item]
    #funcion para extraer las compañias segun su rool
    def extraer_companias(lista, rol):
        if not isinstance(lista, list): return []
        return [
            item['company']['name']
            for item in lista
            if item.get(rol) and 'company' in item and 'name' in item['company']
        ]
    #contador de elemntos de una lista
    def contar_elementos(lista):
        return len(lista) if isinstance(lista, list) else 0
#funcion apra limpiar la url de la aportada para obtener la portada en grande
    def limpiar_cover(cover):
        if not isinstance(cover, dict) or 'url' not in cover: return None
        return f"https:{cover['url'].replace('t_thumb', 't_cover_big')}"


    # Obtener token de IGDB
    token = obtener_token_igdb()
    headers = {'Client-ID': TWITCH_CLIENT_ID, 'Authorization': f'Bearer {token}'}
    igdb_url = "https://api.igdb.com/v4/games"

    #Aqui guardere mi lista de juegos crudos
    juegos_crudos = []
    print(f"Iniciando descarga: {paginas} paginas solicitadas.")

    #ciclo para descargar los juegos por paginas con una pausa para no saturar la api
    for pagina in range(paginas):
        offset = pagina * juegos_por_pagina
        print(f"Consultando pagina {pagina + 1} (Offset: {offset})...")

        query = f"""
        fields name, rating, rating_count, category, parent_game, genres.name, themes.name, 
               game_modes.name, player_perspectives.name, involved_companies.company.name, 
               involved_companies.developer, involved_companies.publisher, franchises.name, 
               first_release_date, dlcs, similar_games, keywords.name, summary, 
               storyline, cover.url, videos, platforms.name;
        where platforms = (6,48,49,130,167,169) & rating_count > 5;
        sort rating_count desc;
        limit {juegos_por_pagina};
        offset {offset};
        """

        response = requests.post(igdb_url, headers=headers, data=query)
        if response.status_code != 200:
            print(f"Error: {response.text}")
            break

        datos = response.json()
        juegos_crudos.extend(datos)
        print(f"Recibidos {len(datos)} registros.")
        
        if pagina < paginas - 1: time.sleep(5)

    if not juegos_crudos: return pd.DataFrame()

    print("Procesando DataFrame")
    df = pd.DataFrame(juegos_crudos)

    # Listas de catalogos
    columnas_lista = ['genres', 'themes', 'game_modes', 'player_perspectives', 'franchises', 'keywords', 'platforms']
    #ciclo para extraer los nombres de los catalogos y guardarlos en nuevas columnas
    for columna in columnas_lista:
        if columna in df.columns:
            df[f'{columna}_list'] = df[columna].apply(extraer_nombres)

    #condicion para extraer los desarrolladores y editores si la columna de compañias esta presente
    if 'involved_companies' in df.columns:
        df['developers_list'] = df['involved_companies'].apply(lambda x: extraer_companias(x, 'developer'))
        df['publishers_list'] = df['involved_companies'].apply(lambda x: extraer_companias(x, 'publisher'))

    #contador de dlcs y videos, y limpieza de la url de la portada
    df['dlc_count'] = df.get('dlcs', pd.Series(dtype='object')).apply(contar_elementos)
    df['video_count'] = df.get('videos', pd.Series(dtype='object')).apply(contar_elementos)
    df['cover_url'] = df.get('cover', pd.Series(dtype='object')).apply(limpiar_cover)
    
    # Asegurar columnas clave con valores por defecto
    df['category'] = df.get('category', 0)
    df['parent_game'] = df.get('parent_game', None)

    # Convertir fecha de lanzamiento de timestamp a date
    if 'first_release_date' in df.columns:
        df['release_date'] = pd.to_datetime(df['first_release_date'], unit='s').dt.date

    # Limpieza final de nulos y valores faltantes
    df = df.assign(
        summary=df.get('summary', pd.Series(dtype='object')).fillna('Sin datos'),
        storyline=df.get('storyline', pd.Series(dtype='object')).fillna('Sin datos'),
        rating=df.get('rating', pd.Series(dtype='float64')).fillna(0.0),
        rating_count=df.get('rating_count', pd.Series(dtype='int64')).fillna(0)
    )
    #elminacion de duplicados por id de juego
    df.drop_duplicates(subset=['id'], inplace=True)
    print(f"Proceso finalizado: {len(df)} registros")
    return df

In [7]:
df_juegos = descargar_y_limpiar_juegos(paginas=30)

Token de IGDB actualizado exitosamente
Iniciando descarga: 30 paginas solicitadas.
Consultando pagina 1 (Offset: 0)...
Recibidos 500 registros.
Consultando pagina 2 (Offset: 500)...
Recibidos 500 registros.
Consultando pagina 3 (Offset: 1000)...
Recibidos 500 registros.
Consultando pagina 4 (Offset: 1500)...
Recibidos 500 registros.
Consultando pagina 5 (Offset: 2000)...
Recibidos 500 registros.
Consultando pagina 6 (Offset: 2500)...
Recibidos 500 registros.
Consultando pagina 7 (Offset: 3000)...
Recibidos 500 registros.
Consultando pagina 8 (Offset: 3500)...
Recibidos 500 registros.
Consultando pagina 9 (Offset: 4000)...
Recibidos 500 registros.
Consultando pagina 10 (Offset: 4500)...
Recibidos 500 registros.
Consultando pagina 11 (Offset: 5000)...
Recibidos 500 registros.
Consultando pagina 12 (Offset: 5500)...
Recibidos 500 registros.
Consultando pagina 13 (Offset: 6000)...
Recibidos 500 registros.
Consultando pagina 14 (Offset: 6500)...
Recibidos 500 registros.
Consultando pagina 1

# Carga a base de datos

In [ ]:

def insertar_en_base_datos(df):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")

    print("Iniciando carga de juegos...")

    #mis columnas principales para la tabla de juegos, el resto se guardara en las tablas de relacion
    columnas_principales = [
        'id', 'name', 'category', 'release_date', 'summary', 'storyline',
        'cover_url', 'rating', 'rating_count', 'dlc_count', 'video_count'
    ]
    
    registros_principales = df[columnas_principales].copy()

    # Asegurar que las fechas sean strings y manejar los nulos
    if 'release_date' in registros_principales.columns:
        registros_principales['release_date'] = registros_principales['release_date'].astype(str)
        registros_principales.loc[registros_principales['release_date'] == 'NaT', 'release_date'] = None

    registros_principales = registros_principales.astype(object).where(pd.notna(registros_principales), None)

    #mi query para insertar los juegos en la tabla principal, con ignore para evitar duplicados por id de igdb
    query_principal = """
        INSERT OR IGNORE INTO CAT_Juego
        (id_igdb, titulo, categoria, fecha_lanzamiento, resumen, historia, url_portada, puntuacion_igdb, conteo_votos_igdb, conteo_dlc, conteo_videos)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
    cursor.executemany(
        query_principal,
        list(registros_principales.itertuples(index=False, name=None))
    )
    conn.commit()

    #crear un diccionario para mapear los id de igdb con los ids internos de la base de datos para las relaciones
    mapa_ids = dict(cursor.execute("SELECT id_igdb, juego_id FROM CAT_Juego").fetchall())

    #funcion apra procesar las rtelaciones de los catalogos
    def procesar_relacion(columna_df, tabla_cat, tabla_rel, col_id_cat):
        df_explod = df[['id', columna_df]].explode(columna_df).dropna(subset=[columna_df])
        if df_explod.empty: return

        valores_unicos = df_explod[columna_df].unique()
        cursor.executemany(f"INSERT OR IGNORE INTO {tabla_cat} (nombre) VALUES (?)", [(v,) for v in valores_unicos])

        mapa_cat = dict(cursor.execute(f"SELECT nombre, {col_id_cat} FROM {tabla_cat}").fetchall())

        puentes = []
        for _, fila in df_explod.iterrows():
            j_id = mapa_ids.get(fila['id'])
            c_id = mapa_cat.get(fila[columna_df])
            if j_id and c_id: puentes.append((j_id, c_id))

        cursor.executemany(f"INSERT OR IGNORE INTO {tabla_rel} (juego_id, {col_id_cat}) VALUES (?, ?)", puentes)

    print("relaciones")
    #mi mapa de ralaciones para iterar sobre los catalogos
    #se para cada tupla de la lista se tiene la columna del df, la tabla de catalogo, la tabla de relacion y el nombre de la columna del id en el catalogo para mapearlo todo
    mapa_relaciones = [
        ('genres_list', 'CAT_Genero', 'REL_Juego_Genero', 'genero_id'),
        ('themes_list', 'CAT_Tematica', 'REL_Juego_Tematica', 'tematica_id'),
        ('game_modes_list', 'CAT_Modo_Juego', 'REL_Juego_Modo', 'modo_id'),
        ('player_perspectives_list', 'CAT_Perspectiva', 'REL_Juego_Perspectiva', 'perspectiva_id'),
        ('franchises_list', 'CAT_Franquicia', 'REL_Juego_Franquicia', 'franquicia_id'),
        ('keywords_list', 'CAT_Etiqueta', 'REL_Juego_Etiqueta', 'etiqueta_id'),
        ('platforms_list', 'CAT_Plataforma', 'REL_Juego_Plataforma', 'plataforma_id'),
        ('developers_list', 'CAT_Empresa', 'REL_Juego_Desarrollador', 'empresa_id'),
        ('publishers_list', 'CAT_Empresa', 'REL_Juego_Editor', 'empresa_id')
    ]

    #ciclo para procesar las relaciones de los catalogos
    for col, t_cat, t_rel, pk in mapa_relaciones:
        if col in df.columns: procesar_relacion(col, t_cat, t_rel, pk)

    #relacionar dlcs con sus juegos principales
    if 'parent_game' in df.columns:
        print("relacionadndo dlc")
        df_dlc = df[['id', 'parent_game']].dropna()
        relaciones_dlc = []

        for _, fila in df_dlc.iterrows():
            id_principal = mapa_ids.get(fila['parent_game'])
            id_dlc = mapa_ids.get(fila['id'])
            if id_principal and id_dlc:
                relaciones_dlc.append((id_principal, id_dlc))

        cursor.executemany("INSERT OR IGNORE INTO REL_Juego_DLC (juego_id_principal, juego_id_dlc) VALUES (?, ?)", relaciones_dlc)

    #relacionar juegos similares
    if 'similar_games' in df.columns:
        df_sim = df[['id', 'similar_games']].explode('similar_games').dropna()
        puentes_sim = [(mapa_ids.get(f['id']), f['similar_games']) for _, f in df_sim.iterrows() if mapa_ids.get(f['id'])]
        cursor.executemany("INSERT OR IGNORE INTO REL_Juegos_Similares (juego_id, id_igdb_similar) VALUES (?, ?)", puentes_sim)

    conn.commit()
    conn.close()
    print("Fin")

In [9]:
insertar_en_base_datos(df_juegos)

Iniciando carga de juegos...
relaciones
relacionadndo dlc
Fin


# Revision rapida